In [1]:
# Importaciones de librerías:

import pandas as pd # Creación de DataFrames
import json # JSON para leer y guardar archivos JSON
import requests # Llamadas a la API de Last.fm
import time # Pausas entre llamadas y no saturar la API

# Conexión a MySQL
import mysql.connector
from mysql.connector import errorcode

In [ ]:
# Fase 1.1. Obtención de los 5 genéros del archivo definitivo.json filtrados de 2020-2025

# Función para filtrar el archivo definitivo.json en función del género y rango de años
def creacion_fichero_genero(genre, año_inicio, año_fin):

    # Abrimos el JSON con todos los registros
    with open("definitivo.json", "r", encoding="utf-8") as f:
        musica_json = json.load(f)

    # Creamos lista vacía para guardar los registros por género
    datos_genero = []

    # Iteramos cada registro del JSON
    for artista in musica_json:

        # Creamos las variables género y año, más fáciles de llamar
        genero = artista["genre"]
        año = artista["year"]
        
        # Aplicamos filtros (género y año)
        if genero == genre and año_inicio <= año <= año_fin:
            
            # Si se cumple el filtro, guardamos la información
            datos_genero.append({
                "id": artista["id"],
                "artist_name": artista["artist_name"],
                "track_name": artista["track_name"],
                "genre": genero,
                "year": año
            })

    # Convertimos la lista de diccionarios en un DataFrame para facilitar la visualización de los datos y exportación
    data_music = pd.DataFrame(datos_genero)

    # Exportamos el DataFrame a JSON por género
    # orient="records" genera lista de diccionarios
    # indent=4 para que sea más legible
    # force_ascii=False mantiene tildes y carácteres especiales
    data_music.to_json(f"data_music_filtrado_{genre}.json", orient="records", indent=4, force_ascii=False)

In [ ]:
# Ejecutamos la función para crear un JSON filtrado por cada género

creacion_fichero_genero("rock", 2020, 2025)

creacion_fichero_genero("latin", 2020, 2025)

creacion_fichero_genero("flamenco", 2020, 2025)

creacion_fichero_genero("rap", 2020, 2025)

creacion_fichero_genero("indie", 2020, 2025)

In [ ]:
# Fase 1.2. Obtención de información de artistas desde la API de Last.fm

# Función que recibirá la API key, el fichero JSON filtrado (fase 1.1) y género
def info_artista_api (api_key, fichero_genero, genre):
    url = "https://ws.audioscrobbler.com/2.0/" # Endpoint base de la API Last.fm

    # Abrimos el archivo JSON con las canciones filtradas por género
    with open(fichero_genero, "r", encoding="utf-8") as f:
        data_music_api = json.load(f) 

    # Creamos lista de artistas únicos
    # Usamos set() para eliminar duplicados y no hacer llamadas repetidas a la API
    artistas_unicos = list(set([ 
        item["artist_name"]
        for item in data_music_api 
        if item["artist_name"]])) 

    # Diccionario donde guardaremos la información obtenida de la API para cada artista
    info_artistas = {}

    # Iteramos por cada cada artista de la lista
    for artista in artistas_unicos:

        # Definimos los parámetros de la llamada a la API
        parametros = {
            "method": "artist.getinfo",   # Método para obtener la biografía
            "artist": artista,            # Nombre del artista
            "api_key": api_key,           # API key
            "format": "json"              # Respuesta en formato JSON
        }

        # Valores por defecto si no encuentra biografía, listeneres, playcount o artistas similares
        biografia = ""
        listeners = ""
        playcount = ""
        similares = ""

        try:
            # Llamada a la API
            # Si la petición tarda +10s, cancelamos con timeout
            response = requests.get(url, params=parametros, timeout=10) 
            # Convertimos la respuesta a JSON
            data_lastfm = response.json() 
        
        except:
            # Si hay error, guardamos valores vacíos para ese artista
            info_artistas[artista] = {
                "biografia": biografia,
                "listeners": listeners,
                "playcount": playcount,
                "similares": similares
            }
            time.sleep(0.05) # Pausa para no saturar la API
            continue # Saltamos al siguiente artista

        # Comprobamos que 'artist' existe en la respuesta
        if "artist" in data_lastfm:
            
            # Creamos una variable más corta, más fácil de usar
            artista_data = data_lastfm["artist"]

            # Extramos biografía, si existe
            if "bio" in artista_data and "summary" in artista_data["bio"]:
                biografia = artista_data["bio"]["summary"]

            # Extraemos estadísticas, si existen 
            if "stats" in artista_data:

                if "listeners" in artista_data["stats"]:
                    listeners = artista_data["stats"]["listeners"]

                if "playcount" in artista_data["stats"]:
                    playcount = artista_data["stats"]["playcount"]

            # Extraemos artistas similares, si existen
            if "similar" in artista_data and "artist" in artista_data["similar"]:
                
                nombres_similares = [] # Creamos una lista vacía para almacenar los artistas similares
                
                # Extraemos cada artista similar que devuelva la API
                for a in artista_data["similar"]["artist"]:
                    nombres_similares.append(a["name"])
                
                # Convertimos la lista en un único string separado por comas
                similares = ", ".join(nombres_similares)

        # Guardamos la información extraida en el diccionario final
        info_artistas[artista] = {
            "biografia": biografia,
            "listeners": listeners,
            "playcount": playcount,
            "similares": similares
        }

        # Pausa para no saturar a la API
        time.sleep(0.05)

    # Guardamos el resultado final en un fichero JSON por género
    with open(f"lastfm_{genre}.json", "w", encoding="utf-8") as f:
        json.dump(info_artistas, f, indent=4, ensure_ascii=False)

In [ ]:
# Ejecutamos función para género 'flamenco' con la API key de Lourdes

api_key_flamenco = "d79b38ce7ccd273a5e94f02f8dd3a299"
info_artista_api(api_key_flamenco, "data_music_filtrado_flamenco.json", "flamenco")

In [ ]:
# Ejecutamos función para género 'indie' con la API key de Ana

api_key_indie = "6786e294b805290158a20e4430160911" # Clave de Last.fm
info_artista_api(api_key_indie, "data_music_filtrado_indie.json", "indie")

In [ ]:
# Ejecutamos función para género 'rock' con la API key de Lorena

api_key_rock = "484e0e00eb4ec9d3093d5fc446733ed0"
info_artista_api(api_key_rock, "data_music_filtrado_rock.json", "rock")

In [ ]:
# Ejecutamos función para género 'rap' con la API key de Laura

api_key_rap = "c46b71050be388c1720fb9983feb9494"
info_artista_api(api_key_rap, "data_music_filtrado_rap.json", "rap")

In [ ]:
# Ejecutamos función para género 'latin' con la API key de Valeria

api_key_latin = "9bb42ab8825458126cac1b7815e878e7"
info_artista_api(api_key_latin, "data_music_filtrado_latin.json", "latin")

In [2]:
# Fase 2. Inserción de datos en la BBDD creada en MySQL

In [ ]:
# Set vacío para guardar los artistas ya introducidos. Así evitamos insertar artistas duplicados entre los ficheros de cada género
artistas_introducidos = set()

# Función para insertar los datos de los artistas en la tabla artistas
def insercion_datos_artistas (nombre_fichero, artistas_introducidos):

    # Lista vacía para guardar los registros insertados
    artistas = []

    # Apertura fichero JSON de Last.fm
    with open(nombre_fichero, "r",encoding="utf-8") as fichero:  
        lastfm_artistas = json.load(fichero)

    # Iteramos cada artista del diccionario JSON
    for nombre, x in lastfm_artistas.items():

        # Comprobamos si el artista no ha sido introducido previamente
        if nombre not in artistas_introducidos:
            # Si no se ha introducido, lo añadimos a la lista
            artistas.append([
                nombre,
                x["biografia"],
                x["listeners"],
                x["similares"],
                x["playcount"]
            ])
            # Guardamos el artista en el set para no repetirlo en los próximos ficheros
            artistas_introducidos.add(nombre) 
    
    # Conexión a MySQL
    cnx = mysql.connector.connect(user="root",
                              password = "AlumnaAdalab",
                              host="127.0.0.1",
                              database= "music_stream",
                              charset='utf8mb4', 
                              read_timeout=300,
                              write_timeout=300)

    # Creación del cursor para ejecutar consultas SQL
    cursor = cnx.cursor()

    # Consulta SQL para insertar artistas
    sql_artistas = """
    INSERT INTO artistas (nombre, biografia,listeners,similares,playcount)
    VALUES (%s, %s, %s, %s, %s)
    """
    
    # Contadores para controlar inserciones y errores
    artistas_insertados = 0
    artistas_errores = 0
    artistas_duplicados = 0

    # Iteramos la lista de artistas que vamos a insertar
    for a in artistas:
        try:
            cursor.execute(sql_artistas,a)
        
        # Prevención de errores
        except mysql.connector.Error as err:
            # Detección de duplicados
            if err.errno == 1062:
                print("Artista duplicado: ", {a[0]})
                artistas_duplicados += 1
                continue
            
            # Detección de errores
            else:
                print(err)
                print("Error Code:", err.errno)
                print("SQLSTATE", err.sqlstate)
                print("Message", err.msg)
                print("Hay un error en la inserción de ", {a[0]})
                artistas_errores += 1
                continue

        # Si no hay error, +1 artista insertado
        else:
            artistas_insertados+=1
    
    # Confirmación de la inserción por fichero
    cnx.commit()
    
    # Mostramos el resumen final
    print("Artistas insertados --->", artistas_insertados)
    print("Artistas con error --->", artistas_errores)
    print("Artistas duplicados --->", artistas_duplicados)

    # Cerramos cursor y conexión
    cursor.close()
    cnx.close()
    print("Se ha cerrado la conexión al servidor")

In [17]:
# Insertando artistas desde el fichero de género 'flamenco'

insercion_datos_artistas("lastfm_flamenco.json", artistas_introducidos)

Artistas insertados ---> 87
Artistas con error ---> 0
Artistas duplicados ---> 0
Se ha cerrado la conexión al servidor


In [18]:
# Insertando artistas desde el fichero de género 'indie'

insercion_datos_artistas("lastfm_indie.json", artistas_introducidos)

Artista duplicado:  {'AURORA'}
Artistas insertados ---> 808
Artistas con error ---> 0
Artistas duplicados ---> 1
Se ha cerrado la conexión al servidor


In [19]:
# Insertando artistas desde el fichero de género 'latin'

insercion_datos_artistas("lastfm_latin.json", artistas_introducidos) 

Artista duplicado:  {'Ovy On the Drums'}
Artistas insertados ---> 563
Artistas con error ---> 0
Artistas duplicados ---> 1
Se ha cerrado la conexión al servidor


In [20]:
# Insertando artistas desde el fichero de género 'rap'

insercion_datos_artistas("lastfm_rap.json", artistas_introducidos) 

Artista duplicado:  {'Амир'}
Artistas insertados ---> 795
Artistas con error ---> 0
Artistas duplicados ---> 1
Se ha cerrado la conexión al servidor


In [21]:
# Insertando artistas desde el fichero de género 'rock'

insercion_datos_artistas("lastfm_rock.json", artistas_introducidos) 

Artistas insertados ---> 680
Artistas con error ---> 0
Artistas duplicados ---> 0
Se ha cerrado la conexión al servidor


In [ ]:
# Función para insertar los datos de canciones en la tabla de canciones
def insercion_datos_canciones (nombre_fichero):

    # Lista vacía para guardar las canciones
    canciones = []

    # Apertura del fichero JSON generado a través de definitivo.JSON
    with open(nombre_fichero, "r",encoding="utf-8") as fichero:  
        listado_canciones = json.load(fichero)

    # Iteramos por cada canción del fichero
    for a in listado_canciones: 
            canciones.append([
                a["id"],
                a["artist_name"],
                a["track_name"], 
                a["genre"],
                a["year"]]
            )
    
    # Conexión a MySQL
    cnx = mysql.connector.connect(user="root",
                              password = "AlumnaAdalab",
                              host="127.0.0.1",
                              database= "music_stream",
                              charset='utf8mb4', # para el formato de caracteres
                              read_timeout=300,
                              write_timeout=300)

    # Creación del cursor para ejecutar consultas
    cursor = cnx.cursor()

    # Consulta SQL para insertar canciones
    sql_canciones = """
    INSERT INTO canciones (id, artist_name, track_name, genre, year)
    VALUES (%s, %s, %s, %s, %s)
    """
    
    # Contadores para controlar inserciones, errores y duplicadas
    canciones_insertadas = 0
    canciones_errores = 0
    canciones_duplicadas = 0

    # Iteramos por cada canción de la lista a insertar
    for x in canciones:
        try:
            cursor.execute(sql_canciones, x)
            #print(a[0]) #1 sería el nombre

        # Prevención de errores
        except mysql.connector.Error as err:

            # Prevención de duplicados
            if err.errno == 1062:
                print(f"Canción duplicada: {x[2]} - {x[1]} - {x[0]}")
                canciones_duplicadas += 1
                continue
            
            # Detección de errores
            else:
                print(err)
                print("Error Code:", err.errno)
                print("SQLSTATE", err.sqlstate)
                print("Message", err.msg)
                print("Hay un error en la inserción de la canción: ", {x[2]})
                canciones_errores += 1
                continue

        # Si no hay error, +1 canción insertada
        else:
            canciones_insertadas+=1

    # Confirmación de la inserción
    cnx.commit()

    # Resumen del proceso
    print("Canciones insertadas --->", canciones_insertadas)
    print("Canciones con error --->", canciones_errores)
    print("Canciones duplicadas --->", canciones_duplicadas)

    # Cerramos cursor y su conexión
    cursor.close()
    cnx.close()
    print("Se ha cerrado la conexión al servidor")

In [23]:
# Insertando canciones desde el fichero de género 'flamenco'

insercion_datos_canciones("data_music_filtrado_flamenco.json") 

Canciones insertadas ---> 295
Canciones con error ---> 0
Canciones duplicadas ---> 0
Se ha cerrado la conexión al servidor


In [24]:
# Insertando canciones desde el fichero de género 'indie'

insercion_datos_canciones("data_music_filtrado_indie.json") 

Canciones insertadas ---> 1852
Canciones con error ---> 0
Canciones duplicadas ---> 0
Se ha cerrado la conexión al servidor


In [25]:
# Insertando canciones desde el fichero de género 'latin'

insercion_datos_canciones("data_music_filtrado_latin.json")  

Canción duplicada: In His Will Your Safe - Earthadox - 54O2LVQ3EEFR5wLZeQdPZY
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Teejayx6 - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Kasher Quon - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Teejayx6 - 1XPzttNdvfOvVuLHFJLJyQ
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Kasher Quon - 1XPzttNdvfOvVuLHFJLJyQ
Canción duplicada: Nostalgia Trap - Mat4yo - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Nostalgia Trap - Kevin Krust - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Nostalgia Trap - Cam Steady - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Chuckie - J-Xtreme - 5ryF0nymvmjD7lVLBST0dp
Canción duplicada: popstar! - shattered - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: popstar! - Waste - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: popstar! - lil eli young bapestah - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: Switch Up - Fat Dolsk - 6HWhbMJshNlFpaUXEJQFgI
Canción duplic

In [26]:
# Insertando canciones desde el fichero de género 'rap'

insercion_datos_canciones("data_music_filtrado_rap.json")  

Canción duplicada: 4K - El Alfa - 1BIXs6CdkPRLytuqoXs6XN
Canción duplicada: Besalo - El Alfa - 2PUdyEnqZUh3zICKE4Fu8N
Canción duplicada: Y Que Fue? - Don Miguelo - 5UcVIU1tsbN7ZsOSpR8AFD
Canción duplicada: Singapur - El Alfa - 5q7icrT1Wn8PHaePIjYYyg
Canción duplicada: Singapur (Remix) - El Alfa - 0lCHRVlRZCEyLj3Gkc1nv1
Canción duplicada: Blin Blin - Bad Gyal - 5u5S7D8NaQzyKkdU4duHJR
Canción duplicada: DELITO - NATHY PELUSO - 0K3dGPXHVALEqW8EEQGc3T
Canción duplicada: 2020 - The Whistlers - 2Prfl6Gv31CGTc6ObzYr4W
Canción duplicada: O' Tope - Gbaskill-D - 5lO3G1VKm7BSoZ6ahlgZ11
Canción duplicada: OVERFLO - J.Broady - 3oWezRR69zSy5m3vLTMpyD
Canción duplicada: In His Will Your Safe - Earthadox - 54O2LVQ3EEFR5wLZeQdPZY
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Teejayx6 - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Kasher Quon - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Teejayx6 - 1XPzttNdvfOvVuLHFJLJyQ
Canción

In [27]:
# Insertando canciones desde el fichero de género 'rock'

insercion_datos_canciones("data_music_filtrado_rock.json") 

Canción duplicada: In His Will Your Safe - Earthadox - 54O2LVQ3EEFR5wLZeQdPZY
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Teejayx6 - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Dynamic Duo 2 (feat. Kasher Quon) - Kasher Quon - 3cnvxdGQcfSARhZXiTzeNn
Canción duplicada: Nostalgia Trap - Mat4yo - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Nostalgia Trap - Kevin Krust - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Nostalgia Trap - Cam Steady - 4Utmmj3BYNRGaQrCGpLSyT
Canción duplicada: Chuckie - J-Xtreme - 5ryF0nymvmjD7lVLBST0dp
Canción duplicada: popstar! - shattered - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: popstar! - Waste - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: popstar! - lil eli young bapestah - 2jssPpYcj3dfmOEbbCxHdf
Canción duplicada: Switch Up - Fat Dolsk - 6HWhbMJshNlFpaUXEJQFgI
Canción duplicada: Nathan Drake Vs Tintin - Freshy Kanal - 2URAgC1IYhpAUa887Kzf9E
Canción duplicada: People - Verónica Romero - 3ShWede9oBhmoRWMxLdz6m
Canción duplicada: Dance Little Sister - Nai